# 01. Случайный лес

**Идея.** Одно дерево решений — набор вопросов вида «в последней реплике больше 40 символов?». Оно легко переобучается.
Случайный лес строит много деревьев, каждое на случайной части диалогов и случайных признаков, а ответ получается
голосованием деревьев. Ошибки отдельных деревьев усредняются.

Порядок: (1) модель по умолчанию как точка отсчёта, (2) случайный поиск параметров на кросс-валидации, (3) выбор числа деревьев,
(4) итог и сохранение, (5) важность признаков, (6) test — один раз в самом конце.

Данные, разбиение и блоки CV — те же, что у старых моделей (первая ячейка).

In [ ]:
import numpy as np
import pandas as pd

import solution_utils as su
from solution_utils import CLASSES, SEED

# Задача 1 из HW1: диалоги с use_reaction == 1 (флаги из dialog_flags.csv), разбиение по group_id
task1 = su.load_task1(su.train_path(), su.flags_path())
train_df, test_df = su.split_task1(task1)          # то же train/test, что в HW1

train_dialogs = train_df[["dialog_id", "turns"]].to_dict("records")
test_dialogs = test_df[["dialog_id", "turns"]].to_dict("records")
y_train, y_test = train_df["reaction"].to_numpy(), test_df["reaction"].to_numpy()
folds = su.get_folds(y_train, train_df["group_id"].to_numpy())   # одни блоки CV для всех моделей

print(f"всего для задачи 1: {len(task1)} | train: {len(train_df)} | test: {len(test_df)}")
pd.DataFrame({"train": pd.Series(y_train).value_counts(normalize=True),
              "test": pd.Series(y_test).value_counts(normalize=True)}).loc[CLASSES].round(3)

## Признаки для дерева

`make_frame` превращает диалог в таблицу: тексты пользователя (все реплики и последняя) и числа. Часть чисел — это флаги из
вашего EDA (код в репликах, зацикливание ассистента), посчитанные по репликам: так они работают и на скрытом тесте, где файла
флагов нет. Деревьям такие готовые числа нужны, длину текста дерево само не посчитает.

In [ ]:
su.make_frame(train_dialogs[:3]).T

## Модель: `Pipeline`

Всё склеено в один объект, чтобы при кросс-валидации TF-IDF обучался только на обучающих блоках (иначе в признаки утечёт
проверочный блок).

- `TfidfVectorizer` — текст в числа по частоте слов (отдельно для всех реплик и для последней);
- `"passthrough"` — числовые колонки идут как есть;
- `class_weight="balanced"` — редкие классы (positive всего ~10%) получают больший вес.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline


def build_pipeline(n_estimators=300):
    features = ColumnTransformer([
        ("all_tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=3000), "user_text"),
        ("last_tfidf", TfidfVectorizer(min_df=3, max_features=1000), "last_user"),
        ("numeric", "passthrough", su.NUMERIC_COLUMNS),
    ])
    forest = RandomForestClassifier(n_estimators=n_estimators, class_weight="balanced", n_jobs=-1, random_state=SEED)
    return Pipeline([("features", features), ("rf", forest)])   # имена шагов важны: на них ссылается код ниже


X_train = su.make_frame(train_dialogs)
X_test = su.make_frame(test_dialogs)

## Шаг 1. Модель по умолчанию

Точка отсчёта. Для сравнения сразу покажем лучшую старую модель из `results/cv_table.csv`.

In [ ]:
history = [{"шаг": "1. по умолчанию", **su.cv_metrics(build_pipeline(), X_train, y_train, folds, n_jobs=1)}]
pd.DataFrame(history).round(3)[["шаг", "macro_f1", "accuracy", "balanced_accuracy", "roc_auc_macro"]]

In [ ]:
old_cv = pd.read_csv("results/cv_table.csv", index_col=0)
old_cv = old_cv[old_cv.index.str.startswith("hw1_")]
print("лучшая старая модель на CV:", old_cv["macro_f1"].idxmax(), round(old_cv["macro_f1"].max(), 3))

## Шаг 2. Подбор параметров (`RandomizedSearchCV`)

Перебрать все комбинации долго, берём 20 случайных. Для каждой — те же 5 блоков `folds`, оценка по `f1_macro`.

| параметр | смысл |
|:--|:--|
| `max_depth` | максимальная глубина дерева; меньше — проще дерево, меньше переобучение |
| `min_samples_leaf` | минимум диалогов в листе; больше — устойчивее |
| `max_features` | сколько признаков смотрит дерево в каждом разбиении; меньше — деревья разнообразнее |
| `class_weight` | способ балансировки классов |
| `max_samples` | какую долю диалогов видит каждое дерево |
| `max_features` у TF-IDF | сколько слов оставить |

Имя вида `шаг__параметр` — так `Pipeline` понимает, к какому шагу относится параметр.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "rf__max_depth": [None, 10, 20, 40],
    "rf__min_samples_leaf": [1, 2, 3, 5, 10],
    "rf__max_features": ["sqrt", "log2", 0.1],
    "rf__class_weight": ["balanced", "balanced_subsample"],
    "rf__max_samples": [None, 0.5, 0.8],
    "features__all_tfidf__max_features": [1000, 3000, 6000],
}

search = RandomizedSearchCV(build_pipeline(), param_distributions, n_iter=20, scoring="f1_macro",
                            cv=folds, random_state=SEED, n_jobs=1, refit=True, verbose=1)
search.fit(X_train, y_train)

results = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cols = ["mean_test_score", "std_test_score"] + [c for c in results.columns if c.startswith("param_")]
results[cols].to_csv("results/rf_search.csv", index=False)       # путь подбора для отчёта
print("лучший macro-F1 на CV:", round(search.best_score_, 3))
print(search.best_params_)
results[cols].head(10).round(3)

Сравнивая верхние и нижние строки, видно, **какие параметры улучшали модель**: это материал для раздела «как подбирали»
(файл `results/rf_search.csv` сохранён).

## Шаг 3. Сколько деревьев

Деревьев больше — результат стабильнее, но дольше. Переобучения от их числа нет: качество выходит на плато.
Берём **наименьшее** число, после которого прироста почти нет.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import cross_val_score

sizes = [50, 100, 200, 300, 500, 800]
scores = []
for n in sizes:
    model = clone(build_pipeline(n)).set_params(**search.best_params_)
    scores.append(cross_val_score(model, X_train, y_train, cv=folds, scoring="f1_macro", n_jobs=1).mean())

plt.plot(sizes, scores, marker="o")
plt.xlabel("число деревьев"); plt.ylabel("macro-F1 (CV)"); plt.grid(alpha=0.3)
plt.savefig("results/rf_n_estimators.png", dpi=120, bbox_inches="tight")
plt.show()

N_TREES = min(n for n, s in zip(sizes, scores) if s >= max(scores) - 0.002)   # наименьшее n в пределах 0.002 от лучшего
print("выбрано деревьев:", N_TREES)

## Шаг 4. Итоговая модель

Метрики на CV считаем так же, как у старых моделей (все метрики из таблицы HW1), записываем строку в общую таблицу,
обучаем на всём train и сохраняем `artifacts/hw2_random_forest.joblib`.

*Оговорка:* параметры подбирались на этих же блоках, поэтому оценка слегка оптимистична. «Честная» оценка — test в конце.
*Заметка на будущее:* в HW1 вы писали, что для скрытого теста модели переобучаете на всех размеченных диалогах (train+test).
Для леса это можно сделать в самом конце, когда выбор модели закончен.

In [ ]:
final = clone(build_pipeline(N_TREES)).set_params(**search.best_params_)
final_row = su.cv_metrics(final, X_train, y_train, folds, n_jobs=1)
history.append({"шаг": "2-3. подбор параметров и числа деревьев", **final_row})
pd.DataFrame(history).round(3).to_csv("results/rf_history.csv", index=False)

cv_table = su.save_result("cv_table.csv", "hw2_random_forest", final_row)
final.fit(X_train, y_train)
print("сохранено:", su.save_model("hw2_random_forest", final))
cv_table.round(3)     # общая таблица: старые модели и случайный лес

In [ ]:
best_old = cv_table[cv_table.index.str.startswith("hw1_")]["macro_f1"].max()
gain = final_row["macro_f1"] - best_old
print(f"лес: {final_row['macro_f1']:.3f} | лучшая старая: {best_old:.3f} | разница: {gain:+.3f}")

## Шаг 5. Важность признаков (MDI)

MDI показывает, насколько признак в среднем уменьшал неоднородность узлов по всем деревьям. Считается быстро, но склонна
завышать важность числовых признаков с большим разбросом значений. Перестановочную важность (`sklearn.inspection.permutation_importance`)
для сравнения сделает тот, кто пишет этот раздел отчёта.

In [ ]:
names = final.named_steps["features"].get_feature_names_out()
importance = pd.Series(final.named_steps["rf"].feature_importances_, index=names)
importance.sort_values(ascending=False).head(15).round(4)

## Шаг 6. Test — запускать один раз

Test нельзя использовать для выбора параметров. Эту ячейку запускаем **один раз**, когда модель окончательно готова.

In [ ]:
from sklearn.metrics import confusion_matrix

proba_test = final.predict_proba(X_test)
test_table = su.save_result("test_table.csv", "hw2_random_forest", su.all_metrics(y_test, proba_test))
test_pred = final.classes_[proba_test.argmax(axis=1)]
display(test_table.round(3))
pd.DataFrame(confusion_matrix(y_test, test_pred, labels=CLASSES),
             index=[f"факт {c}" for c in CLASSES], columns=[f"предсказано {c}" for c in CLASSES])